# Переформулировка разговорных запросов для поиска подкастов

Цель — показать, как контекст диалога и разговорная формулировка влияют на поиск подкастов. Пайплайн rule-based: без обучения моделей и без LLM.

Данные: 78 русскоязычных подкастов и 30 запросов пяти типов. Поиск: TF-IDF по символьным n-граммам и cosine similarity с жёсткими фильтрами.

## 1. Загрузка данных

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd()
if not (ROOT / "data/podcasts_catalog.csv").exists():
    ROOT = Path("/content/media_query_rewriting")
if not (ROOT / "data/podcasts_catalog.csv").exists():
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/LOMELLO/media_query_rewriting.git", str(ROOT)], check=True)

catalog = pd.read_csv(ROOT / "data/podcasts_catalog.csv").fillna("")
queries = pd.read_csv(ROOT / "data/queries.csv").fillna("")
pd.set_option("display.max_colwidth", 80)
print("Подкастов:", len(catalog))
print("Запросов:", len(queries))
display(catalog[["id", "title", "genre", "avg_duration_min", "language", "explicit", "tags"]].head())
display(queries["query_type"].value_counts().rename_axis("Тип").to_frame("Количество"))

Подкастов: 78
Запросов: 30


,id,title,genre,avg_duration_min,language,explicit,tags
0,pod_001,Космическая Подкасточная,Самосовершенствование,73.2,ru,False,без_мата;длинный;документальный;интервью;космос;русский;технологии
1,pod_002,Открытый Космос,Технологии,38.3,ru,False,без_мата;документальный;интервью;космос;русский
2,pod_003,ВРЕМЯ. Пространство. Космос. Введение в гелиоцентрику,Философия,10.4,ru,False,без_мата;документальный;короткий;космос;наука;русский
3,pod_004,Наука. Глубоко,Естественные науки,45.2,ru,False,без_мата;документальный;интервью;наука;русский
4,pod_005,Наука.Легко,Естественные науки,46.4,ru,False,без_мата;документальный;интервью;наука;русский


,Количество
Тип,
уточнение с контекстом,8
настроение/ситуация,6
разговорный/неполный,6
ограничения/исключения,6
не требует переформулировки,4


## 2. Rule-based переформулировка

Правила извлекают тему, настроение, ситуацию, ограничения и исключения. Для follow-up тема и ограничения восстанавливаются из предыдущей реплики. Если запрос уже самостоятельный, пайплайн оставляет его без изменений.

In [2]:
import re
import pandas as pd

TOPIC_ORDER = ['сон', 'космос', 'мозг', 'стартапы', 'криминал', 'политика', 'новости', 'отношения', 'экономика', 'кино', 'музыка', 'юмор', 'литература', 'путешествия', 'спорт', 'дети', 'образование', 'здоровье', 'психология', 'история', 'бизнес', 'технологии', 'наука', 'культура', 'интервью']
TOPIC_PATTERNS = {
    'сон': r'\bсон\b|\bсна\b|\bсны\b|\bсном\b|спать|засн|бессонн|медитац',
    'космос': r'космос|космич|астроном|звезд|звёзд|планет|вселенн',
    'мозг': r'мозг|нейро|психолингвист',
    'стартапы': r'стартап',
    'криминал': r'криминал|преступ|убийств|расследован|детектив|уголовн',
    'политика': r'полит|власт|выбор|государств|войн',
    'новости': r'новост|дайджест',
    'отношения': r'отношен|любов|партнер|партнёр|семейн',
    'экономика': r'экономик|финансов.{0,20}рын|ценн.{0,20}бумаг',
    'кино': r'кино|фильм|сериал|режисс|кинематограф',
    'музыка': r'музык|песн|альбом|музыкант',
    'юмор': r'юмор|смешн|комед|анекдот|стендап|standup|шутк|весёл|весел',
    'литература': r'литератур|книг|писател|поэз|роман',
    'путешествия': r'путешеств|стран|туризм|гид|город',
    'спорт': r'спорт|футбол|трениров|\bбег\b',
    'дети': r'\bдет(и|ей|ям|ьми|ск|ств)|ребен|ребён|подрост|родител',
    'образование': r'образован|школ|универ|учител|лекц|курс',
    'здоровье': r'здоров|медицин|врач|болезн|организм|диет|кож',
    'психология': r'психолог|психотерап|психик|эмоциональн|сознан|самопознан',
    'история': r'истори|древн|цивилизац|импери|археолог|средневек',
    'бизнес': r'бизнес|предприним|компан|маркетинг|менеджмент|финанс|инвест|деньг',
    'технологии': r'технолог|гаджет|\bit\b|айти|нейросет|искусственн.{0,20}интеллект|apple|цифров',
    'наука': r'наук|учён|учен|исследован|физик|биолог|нейронаук',
    'культура': r'культур|искусств|театр|музе',
    'интервью': r'интервью|бесед',
}
TOPIC_ACCUSATIVE = {
    'сон': 'сон', 'космос': 'космос', 'мозг': 'мозг', 'стартапы': 'стартапы',
    'криминал': 'криминал', 'политика': 'политику', 'новости': 'новости',
    'отношения': 'отношения', 'экономика': 'экономику', 'кино': 'кино',
    'музыка': 'музыку', 'юмор': 'юмор', 'литература': 'литературу',
    'путешествия': 'путешествия', 'спорт': 'спорт', 'дети': 'детей',
    'образование': 'образование', 'здоровье': 'здоровье',
    'психология': 'психологию', 'история': 'историю', 'бизнес': 'бизнес',
    'технологии': 'технологии', 'наука': 'науку', 'культура': 'культуру',
    'интервью': 'интервью',
}
CONVERSATIONAL_START = re.compile(r'^(?:ну|а|и|но|может|не знаю|посоветуй|подскажи|хочу|хотелось|нужен|нужна|нужно|что-нибудь|что-то|только|давай)\b')
VAGUE = re.compile(r'залипательн|интересн|что-нибудь|что-то')


def norm(text):
    return re.sub(r'\s+', ' ', str(text).lower().replace('ё', 'е')).strip()


def strip_exclusions(text):
    text = norm(text)
    text = re.sub(r'\bне\s+про\s+[^,.]+', ' ', text)
    text = re.sub(r'\bбез\s+(?:мат[а-я]*|нецензур[а-я]*|воды|болтовни|новост[а-я]*|политик[а-я]*|криминал[а-я]*|отношен[а-я]*|ужас[а-я]*|стендап[а-я]*|новинок)', ' ', text)
    text = re.sub(r'\bне\s+(?:бизнес-новост[а-я]*|новост[а-я]*|политик[а-я]*|стендап[а-я]*|криминал[а-я]*|отношен[а-я]*|ужас[а-я]*|длинн[а-я]*|обзор[а-я]*\s+новинок|новинк[а-я]*)', ' ', text)
    text = re.sub(r'\bкроме\s+[^,.]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def extract_topics(text):
    text = norm(text)
    return [topic for topic in TOPIC_ORDER if re.search(TOPIC_PATTERNS[topic], text)]


def clean_phrase(phrase):
    phrase = norm(phrase)
    phrase = re.split(r'\s+(?:для|с|без|не|до|на|в|во|и)\s+', phrase)[0]
    phrase = re.split(r'[,.;!?]', phrase)[0]
    phrase = re.sub(r'\b(?:есть|пожалуйста|пока|просто)\b', ' ', phrase)
    return re.sub(r'\s+', ' ', phrase).strip()


def extract_topic_phrase(text):
    positive = strip_exclusions(text)
    match = re.search(r'(?:\bпро|\bоб|\bо|\bнасчет|\bна тему)\s+(.+)$', positive)
    if not match:
        return ''
    return clean_phrase(match.group(1))


def extract_style(text):
    text = norm(text)
    styles = []
    if re.search(r'юмористическ|юмор', text):
        styles.append('юмористический')
    elif re.search(r'весел|смешн', text):
        styles.append('весёлый')
    if re.search(r'спокойн|расслаб|тих', text):
        styles.append('спокойный')
    if re.search(r'страшн|жутк|мрачн', text):
        styles.append('страшный')
    if re.search(r'документальн', text):
        styles.append('документальный')
    if re.search(r'научн', text):
        styles.append('научный')
    if re.search(r'развлекательн|залипательн', text):
        styles.append('развлекательный')
    if re.search(r'увлекательн|интересн', text):
        styles.append('увлекательный')
    return list(dict.fromkeys(styles))


def extract_style_phrases(text):
    text = norm(text)
    phrases = []
    if 'без воды' in text:
        phrases.append('без воды')
    if 'без болтовни' in text:
        phrases.append('без болтовни')
    return phrases


def extract_audience(text):
    text = norm(text)
    if re.search(r'для детей|детск|ребен|ребён', text):
        return ['дети']
    return []


def extract_format(text):
    text = norm(text)
    formats = []
    if re.search(r'интервью', text):
        formats.append('интервью')
    if 'документальн' in text:
        formats.append('документальный')
    return formats


def extract_subject(text):
    match = re.search(r'\bс\s+((?:предпринимател|эксперт|гост|учен|учён)[а-я]*)', norm(text))
    return match.group(1) if match else ''


EXCLUDE_PATTERNS = {
    'новости': r'\bне\s+(?:бизнес-новост[а-я]*|новост[а-я]*|обзор[а-я]*\s+новинок|новинк[а-я]*)|без\s+новост[а-я]*',
    'политика': r'\bне\s+про\s+полит[а-я]*|без\s+полит[а-я]*|\bне\s+полит[а-я]*',
    'отношения': r'\bне\s+про\s+отношен[а-я]*|без\s+отношен[а-я]*|\bне\s+отношен[а-я]*',
    'стендап': r'\bне\s+стендап[а-я]*|без\s+стендап[а-я]*',
    'криминал': r'\bне\s+про\s+криминал[а-я]*|без\s+криминал[а-я]*|\bне\s+криминал[а-я]*',
    'ужасы': r'\bне\s+про\s+ужас[а-я]*|без\s+ужас[а-я]*|\bне\s+ужас[а-я]*',
}


def extract_constraints(text):
    text = norm(text)
    constraints = {
        'max_duration_min': None,
        'language': None,
        'explicit_clean': False,
        'excluded_topics': [],
        'audience': [],
        'formats': [],
        'style_phrases': [],
        'styles': [],
    }
    match = re.search(r'до\s*(\d+)\s*(?:мин|минут)', text)
    if match:
        constraints['max_duration_min'] = int(match.group(1))
    match = re.search(r'максимум\s*(\d+)\s*(?:мин|минут)', text)
    if match:
        constraints['max_duration_min'] = int(match.group(1))
    if re.search(r'коротк|покороче|не\s+длинн|не\s+длительн', text):
        constraints['max_duration_min'] = min(constraints['max_duration_min'] or 30, 30)
    if re.search(r'на русском|русскоязыч|по-русски', text):
        constraints['language'] = 'ru'
    if re.search(r'без\s+мат|без\s+нецензур|не\s+мат', text):
        constraints['explicit_clean'] = True
    for topic, pattern in EXCLUDE_PATTERNS.items():
        if re.search(pattern, text):
            constraints['excluded_topics'].append(topic)
    constraints['audience'] = extract_audience(text)
    constraints['formats'] = extract_format(text)
    constraints['style_phrases'] = extract_style_phrases(text)
    constraints['styles'] = extract_style(text)
    return constraints


def extract_situation(text):
    text = norm(text)
    info = {'phrases': [], 'styles': [], 'topics': []}
    if re.search(r'на ночь|перед сном|ко сну', text):
        info['phrases'].append('на ночь')
    if re.search(r'на вечер|вечером', text):
        info['phrases'].append('на вечер')
    if re.search(r'пока\s+мою\s+посуду|уборк|домашн', text):
        info['phrases'].append('для фона')
        info['styles'].append('развлекательный')
    if re.search(r'в дорог|в путь|поездк', text):
        info['phrases'].append('в дорогу')
    if re.search(r'для работы|на работе|во время работы', text):
        info['phrases'].append('для работы')
    if re.search(r'без болтовни', text):
        info['styles'].append('документальный')
    if re.search(r'страшн|жутк|мрачн', text):
        info['topics'].append('криминал')
    return info


def merge_unique(*lists):
    result = []
    for values in lists:
        for value in values:
            if value not in result:
                result.append(value)
    return result


def is_complete(query):
    text = norm(query)
    has_medium = 'подкаст' in text
    has_topic = bool(extract_topics(strip_exclusions(text)) or extract_topic_phrase(text))
    constraints = extract_constraints(text)
    has_constraint = bool(constraints['excluded_topics'] or constraints['audience'])
    starts_conversational = bool(CONVERSATIONAL_START.search(text))
    vague = bool(VAGUE.search(text)) and not has_topic
    return has_medium and (has_topic or has_constraint) and not starts_conversational and not vague


def exclusion_phrase(topic):
    return {
        'новости': 'без новостей',
        'политика': 'без политики',
        'отношения': 'не про отношения',
        'стендап': 'без стендапа',
        'криминал': 'без криминала',
        'ужасы': 'без ужасов',
    }[topic]


def rewrite(query, context=''):
    query = '' if pd.isna(query) else str(query)
    context = '' if pd.isna(context) else str(context)
    text = norm(query)
    if not context and is_complete(query):
        return {
            'original': query,
            'context': context,
            'rewritten': query,
            'rewrite_type': 'без изменений',
            'context_used': False,
            'applied_rules': ['запрос уже самостоятельный'],
            'constraints': extract_constraints(query),
            'assumptions': [],
        }

    context_topics = extract_topics(strip_exclusions(context)) if context else []
    context_phrase = extract_topic_phrase(context) if context else ''
    context_constraints = extract_constraints(context) if context else extract_constraints('')
    context_styles = extract_style(context) if context else []
    context_situations = extract_situation(context) if context else {'phrases': [], 'styles': [], 'topics': []}
    context_subject = extract_subject(context) if context else ''

    current_topics = extract_topics(strip_exclusions(text))
    current_phrase = extract_topic_phrase(text)
    current_constraints = extract_constraints(text)
    current_styles = extract_style(text)
    current_situations = extract_situation(text)
    current_subject = extract_subject(text)

    if re.search(r'страшн[а-я]*\s+истори', text):
        current_topics = [topic for topic in current_topics if topic != 'история']
        current_phrase = ''

    follow_up = bool(re.search(r'^(?:а|и|но|еще|только|давай)\b', text))
    vague = bool(VAGUE.search(text)) and not current_topics and not current_phrase
    use_context = bool(context) and (follow_up or vague or (not current_topics and not current_phrase))

    rules = []
    if context:
        rules.append('учтён контекст диалога')
    if re.search(r'\b(?:ну|вот|блин|короче|типа|просто|не знаю)\b', text):
        rules.append('удалены разговорные маркеры')

    topics = list(current_topics)
    styles = merge_unique(current_styles, current_situations['styles'])
    situations = list(current_situations['phrases'])
    subject = current_subject
    excluded = list(current_constraints['excluded_topics'])
    audience = list(current_constraints['audience'])
    formats = list(current_constraints['formats'])
    style_phrases = list(current_constraints['style_phrases'])
    max_duration = current_constraints['max_duration_min']
    language = current_constraints['language']
    explicit_clean = current_constraints['explicit_clean']
    topic_phrases = [current_phrase] if current_phrase else []

    if use_context:
        rules.append('восстановлена тема из контекста')
        topics = merge_unique(context_topics, topics)
        styles = merge_unique(styles, context_styles)
        if context_phrase and context_phrase not in topic_phrases:
            topic_phrases.insert(0, context_phrase)
        if not subject:
            subject = context_subject
        excluded = merge_unique(excluded, context_constraints['excluded_topics'])
        audience = merge_unique(audience, context_constraints['audience'])
        formats = merge_unique(formats, context_constraints['formats'])
        style_phrases = merge_unique(style_phrases, context_constraints['style_phrases'])
        if max_duration is None:
            max_duration = context_constraints['max_duration_min']
        if language is None:
            language = context_constraints['language']
        explicit_clean = explicit_clean or context_constraints['explicit_clean']

    if current_constraints['max_duration_min'] is not None:
        max_duration = current_constraints['max_duration_min']
        rules.append('сохранено ограничение длительности')
    if current_constraints['language']:
        language = current_constraints['language']
        rules.append('сохранено языковое ограничение')
    if current_constraints['explicit_clean']:
        explicit_clean = True
        rules.append('сохранено ограничение на нецензурную лексику')
    if current_constraints['excluded_topics']:
        rules.append('сохранены исключения')
    if context_constraints['style_phrases'] and not current_constraints['style_phrases']:
        rules.append('перенесено стилевое ограничение из контекста')
    if current_constraints['styles'] or current_situations['styles']:
        rules.append('сохранено настроение/формат')
    if current_constraints['audience']:
        rules.append('сохранено ограничение аудитории')
    if current_constraints['style_phrases']:
        rules.append('сохранены стилевые условия')
    if current_subject:
        rules.append('сохранён адресат интервью')

    topics = merge_unique(topics, current_situations['topics'])
    base = 'подкаст-интервью' if 'интервью' in formats else 'подкаст'
    prefix = []
    for style in styles:
        if style == 'страшный':
            continue
        if style == 'развлекательный' and 'увлекательный' in styles:
            continue
        if style in {'юмористический', 'весёлый', 'развлекательный', 'увлекательный', 'научный', 'документальный', 'спокойный'}:
            prefix.append(style)
    if max_duration is not None and max_duration <= 30:
        if 'короткий' not in prefix:
            prefix.append('короткий')
    if language == 'ru' and 'русскоязычный' not in prefix:
        prefix.insert(0, 'русскоязычный')
    if language == 'ru' and current_constraints['language']:
        rules.append('добавлено языковое ограничение')

    if 'юмористический' in styles or 'весёлый' in styles:
        topics = [topic for topic in topics if topic != 'юмор']
    if 'страшный' in styles or 'криминал' in current_situations['topics']:
        topics = [topic for topic in topics if topic != 'криминал']

    parts = []
    if prefix:
        parts.append(' '.join(dict.fromkeys(prefix)))
    parts.append(base)
    if subject:
        parts.append('с ' + subject)
    if topic_phrases and not subject:
        parts.append('про ' + ' и '.join(topic_phrases))
    elif topics and not subject:
        parts.append('про ' + ' и '.join(TOPIC_ACCUSATIVE[topic] for topic in topics[:2]))
    parts.extend(situations)
    if audience:
        parts.append('для детей')
    if 'страшный' in styles:
        parts.append('со страшными историями')
    parts.extend(style_phrases)
    if max_duration is not None and max_duration <= 30 and re.search(r'до\s*\d+|максимум', text):
        parts.append(f'до {max_duration} минут')
    exclusion_parts = (['без мата'] if explicit_clean else []) + [exclusion_phrase(topic) for topic in excluded]
    rewritten = re.sub(r'\s+', ' ', ' '.join(parts)).strip()
    if exclusion_parts:
        if len(exclusion_parts) == 1:
            phrase = exclusion_parts[0]
            rewritten = rewritten + (', ' + phrase if phrase.startswith('не ') else ' ' + phrase)
        else:
            rewritten = rewritten + ' ' + ' и '.join(exclusion_parts)
    rewritten = re.sub(r'\s+,', ',', rewritten)
    if rewritten in {'', 'подкаст'}:
        rewritten = query

    assumptions = []
    if re.search(r'пока\s+мою\s+посуду|уборк', text):
        assumptions.append('«развлекательный» выведен из бытовой ситуации, а не назван явно')
    if re.search(r'без болтовни', text):
        assumptions.append('«документальный» выведен из пожелания «без болтовни»')
    if 'криминал' in current_situations['topics']:
        assumptions.append('«криминал» выведен из запроса о страшной истории')

    if rewritten == query:
        rewrite_type = 'без изменений'
    elif use_context:
        rewrite_type = 'контекст + переформулировка'
    else:
        rewrite_type = 'нормализация/расширение'

    constraints = {
        'max_duration_min': max_duration,
        'language': language,
        'explicit_clean': explicit_clean,
        'excluded_topics': excluded,
        'audience': audience,
        'formats': formats,
        'style_phrases': style_phrases,
        'styles': styles,
    }
    return {
        'original': query,
        'context': context,
        'rewritten': rewritten,
        'rewrite_type': rewrite_type,
        'context_used': use_context,
        'applied_rules': list(dict.fromkeys(rules)) or ['нормализация формулировки'],
        'constraints': constraints,
        'assumptions': assumptions,
    }

## 3. Результаты переформулировки

In [3]:
rewrite_rows = []
for _, row in queries.iterrows():
    result = rewrite(row["query"], row["context"])
    constraints = result["constraints"]
    rewrite_rows.append({
        "id": row["id"],
        "query": row["query"],
        "context": row["context"],
        "rewritten": result["rewritten"],
        "rewrite_type": result["rewrite_type"],
        "context_used": result["context_used"],
        "applied_rules": "; ".join(result["applied_rules"]),
        "assumptions": "; ".join(result["assumptions"]),
        "max_duration_min": constraints["max_duration_min"],
        "language": constraints["language"],
        "explicit_clean": constraints["explicit_clean"],
        "excluded_topics": "; ".join(constraints["excluded_topics"]),
        "audience": "; ".join(constraints["audience"]),
        "formats": "; ".join(constraints["formats"]),
        "styles": "; ".join(constraints["styles"]),
        "style_phrases": "; ".join(constraints["style_phrases"]),
    })

rewrites = pd.DataFrame(rewrite_rows)
display(rewrites[["id", "query", "context", "rewritten", "rewrite_type", "context_used", "assumptions"]])

,id,query,context,rewritten,rewrite_type,context_used,assumptions
0,m01,Хочу что-нибудь спокойное на ночь,,спокойный подкаст на ночь,нормализация/расширение,False,
1,m02,"Что послушать, пока мою посуду?",,развлекательный подкаст для фона,нормализация/расширение,False,"«развлекательный» выведен из бытовой ситуации, а не назван явно"
2,m03,Посоветуй весёлое в дорогу,,весёлый подкаст в дорогу,нормализация/расширение,False,
3,m04,Хочу страшную историю на вечер,,подкаст на вечер со страшными историями,нормализация/расширение,False,«криминал» выведен из запроса о страшной истории
4,m05,"Нужен подкаст для работы, без болтовни",,документальный подкаст для работы без болтовни,нормализация/расширение,False,«документальный» выведен из пожелания «без болтовни»
5,m06,"Что-то про космос, но не новости",,подкаст про космос без новостей,нормализация/расширение,False,
6,c07,Ну что-нибудь про мозг,,подкаст про мозг,нормализация/расширение,False,
7,c08,а про стартапы есть?,,подкаст про стартапы,нормализация/расширение,False,
8,c09,"что-нибудь интересное, только не длинное",,увлекательный короткий подкаст,нормализация/расширение,False,
9,c10,"посоветуй подкаст, чтобы залипательно",,развлекательный подкаст,нормализация/расширение,False,


## 4. Примеры

In [4]:
example_ids = ["m02", "m05", "f13", "f16", "x25", "n28"]
display(rewrites.set_index("id").loc[example_ids, ["query", "context", "rewritten", "rewrite_type", "assumptions"]])

,query,context,rewritten,rewrite_type,assumptions
id,,,,,
m02,"Что послушать, пока мою посуду?",,развлекательный подкаст для фона,нормализация/расширение,"«развлекательный» выведен из бытовой ситуации, а не назван явно"
m05,"Нужен подкаст для работы, без болтовни",,документальный подкаст для работы без болтовни,нормализация/расширение,«документальный» выведен из пожелания «без болтовни»
f13,А покороче?,Ищу подкаст про космос без воды.,короткий подкаст про космос без воды,контекст + переформулировка,
f16,Давай про древние цивилизации,Нужен подкаст про историю.,подкаст про историю и древние цивилизации,контекст + переформулировка,
x25,Посоветуй спокойный подкаст без мата и без криминала,,спокойный подкаст без мата и без криминала,нормализация/расширение,
n28,Подкаст про психологию сна без мата,,Подкаст про психологию сна без мата,без изменений,


## 5. Поиск

Сравниваются два режима: исходный запрос с ограничениями, извлечёнными из него, и переформулированный запрос с объединёнными ограничениями.

In [5]:
catalog["search_text"] = (catalog["title"] + " " + catalog["genre"] + " " + catalog["description"] + " " + catalog["tags"].str.replace(";", " ", regex=False)).map(norm)

vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1, sublinear_tf=True)
matrix = vectorizer.fit_transform(catalog["search_text"])

TAG_MAP = {
    "новости": "новости",
    "политика": "политика",
    "отношения": "отношения",
    "стендап": "стендап",
    "криминал": "криминал",
    "ужасы": "ужасы",
}

def allowed(row, constraints):
    if constraints["max_duration_min"] is not None and row["avg_duration_min"] > constraints["max_duration_min"]:
        return False
    if constraints["language"] == "ru" and not str(row["language"]).startswith("ru"):
        return False
    if constraints["explicit_clean"] and bool(row["explicit"]):
        return False
    if constraints["audience"] and "для_детей" not in str(row["tags"]).split(";"):
        return False
    for topic in constraints["excluded_topics"]:
        tag = TAG_MAP.get(topic)
        if tag and tag in str(row["tags"]).split(";"):
            return False
    return True

def search(text, constraints, n=5):
    query_vector = vectorizer.transform([norm(text)])
    scores = cosine_similarity(query_vector, matrix).ravel()
    candidates = []
    for index, score in enumerate(scores):
        row = catalog.iloc[index]
        if allowed(row, constraints):
            candidates.append((index, float(score)))
    candidates.sort(key=lambda item: item[1], reverse=True)
    results = []
    for index, score in candidates[:n]:
        row = catalog.iloc[index]
        results.append({
            "id": str(row["id"]),
            "title": row["title"],
            "score": round(score, 3),
            "tags": row["tags"],
            "duration": float(row["avg_duration_min"]),
            "explicit": bool(row["explicit"]),
            "allowed": allowed(row, constraints),
        })
    return results

## 6. Оценка до и после

- **Hit@5** — есть ли в топ-5 подкаст с целевым eval-тегом.
- **P@5** — доля подкастов с целевым тегом в топ-5.
- **Сохранение темы** — целевая тема присутствует в тексте запроса по словарю синонимов.
- **Сохранение ограничений** — все ожидаемые ограничения извлечены правилами.

In [6]:
ALIASES = {
    "сон": ["сон", "сна", "спокой", "ночь", "медитац"],
    "космос": ["космос", "космич", "астроном", "планет", "вселенн"],
    "развлечение": ["развлека", "увлека", "весел", "смеш", "юмор"],
    "развлекательный": ["развлека", "увлека"],
    "юмор": ["юмор", "весел", "смеш"],
    "криминал": ["криминал", "страшн", "преступ"],
    "документальный": ["документальн"],
    "мозг": ["мозг", "нейро"],
    "стартапы": ["стартап"],
    "короткий": ["коротк", "до 30", "не длинн", "покороче"],
    "история": ["истори", "древн", "цивилизац"],
    "психология": ["психолог", "психотерап"],
    "бизнес": ["бизнес", "предприним"],
    "здоровье": ["здоров", "медицин"],
    "технологии": ["технолог", "гаджет", "it", "айти", "нейросет"],
    "экономика": ["экономик", "финанс"],
    "кино": ["кино", "фильм", "сериал"],
    "интервью": ["интервью"],
    "спокойный": ["спокойн", "расслаб"],
}

def covered(text, topic):
    text = norm(text)
    return any(norm(alias) in text for alias in ALIASES[topic])

def satisfied(name, constraints, text):
    name = name.strip()
    if name == "спокойный":
        return "спокойный" in constraints["styles"]
    if name == "короткий":
        return constraints["max_duration_min"] is not None and constraints["max_duration_min"] <= 30
    if name == "развлекательный":
        return "развлекательный" in constraints["styles"]
    if name == "весёлый":
        return "весёлый" in constraints["styles"]
    if name == "страшный":
        return "страшный" in constraints["styles"]
    if name == "документальный":
        return "документальный" in constraints["formats"] or "документальный" in constraints["styles"]
    if name == "без болтовни":
        return "без болтовни" in constraints["style_phrases"]
    if name in {"не новости", "не новинки"}:
        return "новости" in constraints["excluded_topics"]
    if name == "не отношения":
        return "отношения" in constraints["excluded_topics"]
    if name == "научный":
        return "научный" in constraints["styles"]
    if name == "не стендап":
        return "стендап" in constraints["excluded_topics"]
    if name == "русский":
        return constraints["language"] == "ru"
    if name == "без мата":
        return bool(constraints["explicit_clean"])
    if name == "не политика":
        return "политика" in constraints["excluded_topics"]
    if name == "для детей":
        return "дети" in constraints["audience"]
    if name == "до 30 минут":
        return constraints["max_duration_min"] is not None and constraints["max_duration_min"] <= 30
    if name == "не криминал":
        return "криминал" in constraints["excluded_topics"]
    if name == "интервью":
        return "интервью" in constraints["formats"]
    if name == "без воды":
        return "без воды" in constraints["style_phrases"]
    return False

def split_constraints(text):
    return [item.strip() for item in str(text).split(";") if item.strip()]

evaluation_rows = []
for _, row in queries.iterrows():
    result = rewrite(row["query"], row["context"])
    before_constraints = extract_constraints(row["query"])
    after_constraints = result["constraints"]
    before = search(row["query"], before_constraints)
    after = search(result["rewritten"], after_constraints)
    eval_tag = row["eval_tag"]
    before_hits = [eval_tag in str(item["tags"]).split(";") for item in before]
    after_hits = [eval_tag in str(item["tags"]).split(";") for item in after]
    expected_constraints = split_constraints(row["expected_constraints"])
    evaluation_rows.append({
        "id": row["id"],
        "query_type": row["query_type"],
        "query": row["query"],
        "context": row["context"],
        "rewritten": result["rewritten"],
        "rewrite_type": result["rewrite_type"],
        "context_used": result["context_used"],
        "topic_raw": covered(row["query"], row["expected_topic"]),
        "topic_after": covered(result["rewritten"], row["expected_topic"]),
        "constraint_raw": all(satisfied(item, before_constraints, row["query"]) for item in expected_constraints),
        "constraint_after": all(satisfied(item, after_constraints, result["rewritten"]) for item in expected_constraints),
        "before_hit": any(before_hits),
        "after_hit": any(after_hits),
        "before_p5": round(float(np.mean(before_hits)), 4),
        "after_p5": round(float(np.mean(after_hits)), 4),
        "before_top1": before[0]["title"] if before else "",
        "after_top1": after[0]["title"] if after else "",
        "before_results": before,
        "after_results": after,
    })

evaluation = pd.DataFrame(evaluation_rows)

context_rows = evaluation[evaluation["context"] != ""]
non_context_rows = evaluation[evaluation["context"] == ""]
noop_rows = evaluation.loc[~queries["needs_rewrite"].astype(bool)]

quality = pd.DataFrame({
    "Проверка": [
        "Тема сохранена",
        "Ограничения сохранены",
        "Контекст учтён в follow-up",
        "Контекст не подмешан без необходимости",
        "No-op оставлен без изменений",
    ],
    "Результат": [
        f"{int(evaluation['topic_after'].sum())}/{len(evaluation)}",
        f"{int(evaluation['constraint_after'].sum())}/{len(evaluation)}",
        f"{int(context_rows['context_used'].sum())}/{len(context_rows)}",
        f"{int((~non_context_rows['context_used']).sum())}/{len(non_context_rows)}",
        f"{int((noop_rows['rewrite_type'] == 'без изменений').sum())}/{len(noop_rows)}",
    ],
})
display(quality)

retrieval = pd.DataFrame({
    "Метрика": ["Hit@5", "P@5"],
    "До": [f"{evaluation['before_hit'].mean():.3f}", f"{evaluation['before_p5'].mean():.3f}"],
    "После": [f"{evaluation['after_hit'].mean():.3f}", f"{evaluation['after_p5'].mean():.3f}"],
})
display(retrieval)

before_hit_count = int(evaluation["before_hit"].sum())
after_hit_count = int(evaluation["after_hit"].sum())
print(f"Запросов с попаданием в топ-5: до {before_hit_count}/{len(evaluation)}, после {after_hit_count}/{len(evaluation)}")
print(f"Запросов с сохранённой темой: до {int(evaluation['topic_raw'].sum())}/{len(evaluation)}, после {int(evaluation['topic_after'].sum())}/{len(evaluation)}")
print(f"Запросов с сохранёнными ограничениями: до {int(evaluation['constraint_raw'].sum())}/{len(evaluation)}, после {int(evaluation['constraint_after'].sum())}/{len(evaluation)}")

,Проверка,Результат
0,Тема сохранена,30/30
1,Ограничения сохранены,30/30
2,Контекст учтён в follow-up,8/8
3,Контекст не подмешан без необходимости,22/22
4,No-op оставлен без изменений,5/5


,Метрика,До,После
0,Hit@5,0.900,0.967
1,P@5,0.523,0.790


Запросов с попаданием в топ-5: до 27/30, после 29/30
Запросов с сохранённой темой: до 20/30, после 30/30
Запросов с сохранёнными ограничениями: до 27/30, после 30/30


## 7. Результаты по типам запросов

In [7]:
by_type = evaluation.groupby("query_type").agg(
    Запросов=("id", "count"),
    **{
        "Hit@5 до": ("before_hit", "mean"),
        "Hit@5 после": ("after_hit", "mean"),
        "P@5 до": ("before_p5", "mean"),
        "P@5 после": ("after_p5", "mean"),
    },
).round(3)
display(by_type)

,Запросов,Hit@5 до,Hit@5 после,P@5 до,P@5 после
query_type,,,,,
настроение/ситуация,6,1.00,1.00,0.433,0.767
не требует переформулировки,4,0.75,0.75,0.650,0.650
ограничения/исключения,6,1.00,1.00,0.817,0.817
разговорный/неполный,6,1.00,1.00,0.633,0.867
уточнение с контекстом,8,0.75,1.00,0.225,0.800


## 8. Ошибка поиска

In [8]:
display(evaluation.loc[~evaluation["after_hit"], ["id", "query", "rewritten", "after_top1"]])
n28 = evaluation.set_index("id").loc["n28"]
display(pd.DataFrame(n28["after_results"])[["title", "tags", "score"]])

,id,query,rewritten,after_top1
27,n28,Подкаст про психологию сна без мата,Подкаст про психологию сна без мата,Психология. Погружение в себя


,title,tags,score
0,Психология. Погружение в себя,без_мата;мозг;психология;русский,0.236
1,Психология для жизни,без_мата;короткий;мозг;психология;русский,0.223
2,Pro отношения,без_мата;отношения;психология;русский,0.171
3,Осознанное родительство: Отцы и дети,без_мата;дети;для_детей;интервью;образование;психология;русский,0.151
4,Хвост виляет собакой | Поведенческая экономика | Психология поведения,без_мата;бизнес;короткий;наука;психология;русский;экономика,0.149


## 9. Сохранение результатов

In [9]:
rewrite_columns = [
    "id", "query", "context", "rewritten", "rewrite_type", "context_used",
    "applied_rules", "assumptions", "max_duration_min", "language",
    "explicit_clean", "excluded_topics", "audience", "formats", "styles", "style_phrases",
]
rewrites[rewrite_columns].to_csv(ROOT / "data/rewritten_queries.csv", index=False)

comparison_columns = [
    "id", "query", "rewritten", "before_hit", "after_hit", "before_p5", "after_p5",
    "before_top1", "after_top1", "before_results", "after_results",
]
comparison = evaluation[comparison_columns].copy()
comparison["before_results"] = comparison["before_results"].map(lambda value: json.dumps(value, ensure_ascii=False))
comparison["after_results"] = comparison["after_results"].map(lambda value: json.dumps(value, ensure_ascii=False))
comparison.to_csv(ROOT / "data/search_comparison.csv", index=False)
print("Сохранено: data/rewritten_queries.csv")
print("Сохранено: data/search_comparison.csv")

Сохранено: data/rewritten_queries.csv
Сохранено: data/search_comparison.csv


## 10. Выводы и ограничения

Переформулировка заметно улучшает поиск для уточнений с контекстом: P@5 растёт с 0.22 до 0.80. Для настроения и ситуации P@5 растёт с 0.43 до 0.77. Все 30 запросов сохраняют целевую тему и ограничения, 8 follow-up корректно используют контекст, 5 самостоятельных запросов остаются no-op. Общий Hit@5 — 27/30 до и 29/30 после, P@5 — 0.523 до и 0.790 после.

Единственный промах после переформулировки — `n28`: запрос уже самостоятельный, поэтому остаётся без изменений, но char n-gram поиск ставит на первое место общую психологию, а не сон. Это ограничение лексического поиска, а не ошибка правил.

Часть решений является выводом из контекста: `m02` получает «развлекательный» из ситуации с мытьём посуды, `m05` — «документальный» из пожелания «без болтовни», `m04` — «страшный» из запроса о страшной истории. Такие выводы полезны, но их следует считать предположениями.

Ограничения: небольшой вручную размеченный каталог, синтетические запросы, один eval-тег на запрос, отсутствие пользовательской оценки и чувствительность char n-gram поиска к морфологии. Для развития можно добавить семантические эмбеддинги, расширение синонимов и ручную проверку релевантности.

## 11. Итог

Rule-based подход прозрачен и воспроизводим: он улучшает retrieval без обучения и сохраняет ограничения. Основной прирост даёт восстановление темы из контекста; основной риск — предположения, которые правила добавляют из ситуации.